# Exercise - Digitization and Data Analytics - Machine Learning

Application for the material from the lecture and bit of fun with a real-world data

In [ ]:
!pip install pandas
!pip install numpy
!pip install scikit-learn
!pip install matplotlib
!pip install seaborn

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing, fetch_covtype
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, r2_score, mean_squared_error
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# Load Datasets
california = fetch_california_housing()
cover_type = fetch_covtype()

# Preparing the Dataset with Numerical Target variable (Regression)
reg_data_without_target = pd.DataFrame(california.data, columns=california.feature_names)
reg_target = pd.DataFrame(california.target, columns=['House_Value'])

# Preparing the Dataset with Categorical Target variable (Classification)
cla_data_without_target = pd.DataFrame(cover_type.data, columns=cover_type.feature_names)
cla_target = pd.DataFrame(cover_type.target, columns=['Cover_Type'])

## Task 1 - Correlation Matrices

### Compute the correlation matrices for the datasets and plot it to visualize the results.

**What is it?** A correlation matrix is a table that shows how strongly different variables are linked to one another. 

**Why are we doing this?** We want to see if some of our columns are basically giving us the exact same information (overlapping). If they overlap too much, it means we can safely compress our dataset in the next steps.

### Observing the the plots, what variables are strongly correlated?

In [ ]:
# California Housing Correlation Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(reg_data_without_target.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("California Housing Correlation Matrix")
plt.show()

# Cover Type Correlation Matrix (Showing the first 10 continuous features for readability)
plt.figure(figsize=(10, 8))
sns.heatmap(cla_data_without_target.iloc[:, :10].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Cover Type Correlation Matrix (First 10 Features)")
plt.show()

## Task 2 - Principle Component Analysis (PCA)


**What is it?** PCA is a linear mathematical transformation that squishes our data down into fewer, non-correlated columns (components) while trying to keep the original variance (information) intact.

**Why are we doing this?** To remove noise, solve the multicollinearity issue we found in Task 1, and make our downstream models run faster.

### Apply PCA (Principle Component Analysis) to the datasets.

In [ ]:
# CRITICAL: PCA is highly sensitive to the scale of the data. 
# We MUST standardize the data (mean = 0, variance = 1) before applying PCA.
scaler_reg = StandardScaler()
reg_data_scaled = scaler_reg.fit_transform(reg_data_without_target)

# Initialize and fit PCA on the scaled regression data
pca_reg = PCA()
pca_reg.fit(reg_data_scaled)

# Repeat the scaling and PCA fitting for the classification data
scaler_cla = StandardScaler()
cla_data_scaled = scaler_cla.fit_transform(cla_data_without_target)

pca_cla = PCA()
pca_cla.fit(cla_data_scaled)

# Plotting the Scree Plots to visualize explained variance
plt.figure(figsize=(10, 4))

# Scree plot for California Housing
plt.subplot(1, 2, 1)
# np.cumsum calculates the cumulative sum of the variance explained by each consecutive component
plt.plot(range(1, len(pca_reg.explained_variance_ratio_) + 1), np.cumsum(pca_reg.explained_variance_ratio_), marker='o')
# Draw a red dashed line at 90% to help us find our threshold
plt.axhline(y=0.90, color='r', linestyle='--')
plt.title('Scree Plot - California Housing')
plt.xlabel('Number of PCs (q)')
plt.ylabel('Cumulative Explained Variance')

# Scree plot for Cover Type
plt.subplot(1, 2, 2)
plt.plot(range(1, len(pca_cla.explained_variance_ratio_) + 1), np.cumsum(pca_cla.explained_variance_ratio_))
plt.axhline(y=0.90, color='r', linestyle='--')
plt.title('Scree Plot - Cover Type')
plt.xlabel('Number of PCs (q)')
plt.ylabel('Cumulative Explained Variance')
plt.tight_layout()
plt.show()

### How many Principle Components are sufficient for the given datasets?

## Task 3 - t-SNE

### Apply the t-SNE method to the data 



**What is it?** t-SNE is a non-linear algorithm that maps high-dimensional data into a 2D or 3D space, grouping similar data points close together.
 
**Why are we doing this?** PCA is great for models, but t-SNE is purely for our human eyes. We use it to visually explore if there are hidden, non-linear clusters in our data.

In [ ]:
# t-SNE is highly computationally intensive (O(N^2) complexity). 
# To prevent the notebook from crashing during class, we use a random subset of 2000 rows.
sample_size = 2000

# Initialize t-SNE for 2 dimensions to plot on a standard X/Y axis
tsne_reg = TSNE(n_components=2, random_state=42)
tsne_reg_results = tsne_reg.fit_transform(reg_data_scaled[:sample_size])

plt.figure(figsize=(12, 5))

# Scatter plot for Regression Data
plt.subplot(1, 2, 1)
# Color the points (c=...) based on the actual house value to see if expensive houses cluster together
plt.scatter(tsne_reg_results[:, 0], tsne_reg_results[:, 1], c=reg_target['House_Value'].values[:sample_size], cmap='viridis', alpha=0.6)
plt.colorbar(label='House Value')
plt.title('t-SNE - California Housing')

# Initialize and apply t-SNE to Classification Data
tsne_cla = TSNE(n_components=2, random_state=42)
tsne_cla_results = tsne_cla.fit_transform(cla_data_scaled[:sample_size])

# Scatter plot for Classification Data
plt.subplot(1, 2, 2)
# Color the points based on their actual cover type class
plt.scatter(tsne_cla_results[:, 0], tsne_cla_results[:, 1], c=cla_target['Cover_Type'].values[:sample_size], cmap='tab10', alpha=0.6)
plt.colorbar(label='Cover Type')
plt.title('t-SNE - Cover Type')
plt.show()

## Task 4 - Linear Regression



**What is it?** Linear Regression tries to draw the best possible straight line through our data points to predict a continuous number.

**Why are we doing this?** We will predict housing prices. We will compare a standard model against a model stripped of bad variables (RFE), and against models trained on our PCA/t-SNE compressed data.

### Build a Linear Regression model for the housing dataset, ie, `reg_data`

In [ ]:
# Split the data into 80% training data and 20% testing data to prevent data leakage and overfitting
X_train, X_test, y_train, y_test = train_test_split(
    reg_data_without_target, reg_target, test_size=0.2, random_state=42)

# 1. Standard Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)          # Model learns from the training data
y_pred = lr.predict(X_test)       # Model makes predictions on the unseen test data

# Evaluate Goodness of Fit
print(f"Standard Linear Regression - R^2: {r2_score(y_test, y_pred):.4f}, MSE: {mean_squared_error(y_test, y_pred):.4f}")

# 2. Backward Variable Selection using RFE (Recursive Feature Elimination)
# RFE iteratively removes the weakest features until only the specified number (5) remains
rfe_lr = RFE(estimator=LinearRegression(), n_features_to_select=5)
rfe_lr.fit(X_train, y_train)
y_pred_rfe = rfe_lr.predict(X_test)

print(f"RFE Reduced Model - R^2: {r2_score(y_test, y_pred_rfe):.4f}")
# Print the boolean mask to show which columns were kept
print(f"Selected Features: {X_train.columns[rfe_lr.support_].tolist()}")

# 3. PCA Regression (Principal Component Regression)
pca_lr = PCA(n_components=5)
# IMPORTANT: We only FIT the scaler and PCA on the training data, then merely TRANSFORM the test data
X_train_pca = pca_lr.fit_transform(scaler_reg.fit_transform(X_train))
X_test_pca = pca_lr.transform(scaler_reg.transform(X_test))

lr_pca = LinearRegression()
lr_pca.fit(X_train_pca, y_train)
y_pred_pca = lr_pca.predict(X_test_pca)
print(f"PCA Regression - R^2: {r2_score(y_test, y_pred_pca):.4f}")

# 4. t-SNE Regression 
# Since we only generated t-SNE for 2000 points, we split that small subset
X_tsne_train, X_tsne_test, y_tsne_train, y_tsne_test = train_test_split(
    tsne_reg_results, reg_target[:sample_size], test_size=0.2, random_state=42)

lr_tsne = LinearRegression()
lr_tsne.fit(X_tsne_train, y_tsne_train)
y_pred_tsne = lr_tsne.predict(X_tsne_test)
print(f"t-SNE Regression - R^2: {r2_score(y_tsne_test, y_pred_tsne):.4f}")

Is it something wrong here?

## Task 5 - Logistic Regression

### Build a Logistic Regression model for the cover type dataset, ie, `cla_data`

## Task 5 - Logistic Regression

**What is it?** Logistic Regression calculates the probability that a data point belongs to a certain discrete category using a sigmoid function.

**Why are we doing this?** To sort our forest data into classes (Cover Types). We will evaluate its goodness of fit using a confusion matrix.

In [ ]:
# Split the classification data
# .values.ravel() flattens the target array, which Logistic Regression expects
X_cla_train, X_cla_test, y_cla_train, y_cla_test = train_test_split(
    cla_data_without_target, cla_target.values.ravel(), test_size=0.2, random_state=42)



In [ ]:
# 1. Standard Logistic Regression
# max_iter=500 gives the gradient descent algorithm enough time to converge
log_reg = LogisticRegression(max_iter=200)
log_reg.fit(X_cla_train, y_cla_train)
y_cla_pred = log_reg.predict(X_cla_test)

print(f"Standard Logistic Regression Accuracy: {accuracy_score(y_cla_test, y_cla_pred):.4f}")



In [ ]:
# Generate a Confusion Matrix to visualize True Positives, False Positives, etc.
cm = confusion_matrix(y_cla_test, y_cla_pred)
plt.figure(figsize=(8, 6))
# We slice [:7, :7] because there are 7 cover types
sns.heatmap(cm[:7, :7], annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix (Goodness of Fit)")
plt.ylabel('Actual Class (Y)')
plt.xlabel('Predicted Class (Y-hat)')
plt.show()



In [ ]:
# 2. RFE Logistic Regression
# To save time in class, we fit RFE on a 5000-row subset
rfe_log = RFE(estimator=LogisticRegression(max_iter=200), n_features_to_select=15)
rfe_log.fit(X_cla_train[:5000], y_cla_train[:5000]) 
y_cla_pred_rfe = rfe_log.predict(X_cla_test)
print(f"RFE Logistic Regression Accuracy: {accuracy_score(y_cla_test, y_cla_pred_rfe):.4f}")



In [ ]:
# 3. PCA Logistic Regression
# Compress down to 15 components
pca_cla_model = PCA(n_components=15)
X_train_cla_pca = pca_cla_model.fit_transform(scaler_cla.fit_transform(X_cla_train))
X_test_cla_pca = pca_cla_model.transform(scaler_cla.transform(X_cla_test))

log_reg_pca = LogisticRegression(max_iter=200)
log_reg_pca.fit(X_train_cla_pca, y_cla_train)
y_cla_pred_pca = log_reg_pca.predict(X_test_cla_pca)
print(f"PCA Logistic Regression Accuracy: {accuracy_score(y_cla_test, y_cla_pred_pca):.4f}")



In [ ]:
# 4. t-SNE Logistic Regression (using the 2000-row subset)
X_tsne_cla_train, X_tsne_cla_test, y_tsne_cla_train, y_tsne_cla_test = train_test_split(
    tsne_cla_results, cla_target[:sample_size].values.ravel(), test_size=0.2, random_state=42)

log_reg_tsne = LogisticRegression(max_iter=200)
log_reg_tsne.fit(X_tsne_cla_train, y_tsne_cla_train)
y_cla_pred_tsne = log_reg_tsne.predict(X_tsne_cla_test)
print(f"t-SNE Logistic Regression Accuracy: {accuracy_score(y_tsne_cla_test, y_cla_pred_tsne):.4f}")

## Fun time! 

Apply what you studied before about regression on the real data! Use leftovers_milk_data.csv data and try to predict the number for the next day.